# Initialization

In [0]:

import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

In [0]:
CATALOG = "humanoid_robotics"
BRONZE  = f"{CATALOG}.bronze"
SILVER  = f"{CATALOG}.silver"

SOURCE_TABLES = {
    "company_seed":  f"{BRONZE}.company_seed",
    "company_other": f"{BRONZE}.company_other",
}

TARGET_TABLE = f"{SILVER}.company"

VALID_COMPANY_TYPES = ["humanoid_maker", "tier1", "investor",
                       "customer", "technology_partner", "other"]


def clean_str(col_name):
    c = F.trim(F.col(col_name))
    return F.when(c == "", None).otherwise(c)


def yn_to_bool(col_name):
    return F.upper(clean_str(col_name)) == "Y"

# Read Bronze Table

In [0]:
dfs = []
for source_name, table in SOURCE_TABLES.items():
    df = spark.table(table).withColumn("seed_source", F.lit(source_name))
    print(f"{table}: {df.count()} rows")
    dfs.append(df)

raw_df = dfs[0]
for df in dfs[1:]:
    raw_df = raw_df.unionByName(df)

In [0]:
raw_df.display()

# Silver Transformations

### Trimming

In [0]:
# Trim every string column. Start from raw_df once and keep building on df;
# (before: each loop started again from raw_df, so only the last string column was trimmed)
df = raw_df
for field in raw_df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df = (
    df
    .withColumn("company_id",   F.lower("company_id"))
    .withColumn("company_type", F.lower("company_type"))
    .withColumn("founded_year", F.col("founded_year").cast("int"))
    .withColumn("is_listed",    F.col("is_listed") == "Y")
    .withColumn("is_core",      F.col("is_core") == "Y")
    .withColumn("verified",     F.col("verified") == "Y")
    )



In [0]:
df.display()

# Write into Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable(TARGET_TABLE)

In [0]:
%sql

select * from humanoid_robotics.silver.company;